# TC-26 L–H Threshold Scaling Analysis
Reproduces the TC-26(B_t) scaling from Delabie *et al.* 2026, Nucl. Fusion 66 036016 from IMAS migrated DB

In [1]:
import os
import numpy as np
import imas
import scipy as sp

ROOT = os.path.dirname(os.getcwd())
TC26_DIR = os.path.join(ROOT, "resources", "results", "tc26")

pulse_dirs = sorted(
    os.path.join(TC26_DIR, d)
    for d in os.listdir(TC26_DIR)
    if d.startswith("pulse_")
)
N = len(pulse_dirs)
print(f"{N} pulses in {TC26_DIR}")

689 pulses in c:\Users\curranf\IDStools\resources\results\tc26


## Load scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| PLTH | `summary/global_quantities/power_loss/value` | W |
| SPLASMA | `equilibrium/time_slice/global_quantities/surface` | m^2 |
| BT | `summary/global_quantities/b0/value` | T |
| NEL | `summary/line_average/n_e/value` | m^-3 |
| PGASA (M_eff) | `summary/volume_average/meff_hydrogenic/value` | AMU |
| IP | `summary/global_quantities/ip/value` | A |
| RGEO | `summary/global_quantities/r0/value` | m |
| DIVCON | `divertors/divertor(0)/description` | — |
| TOK | `summary/machine` | — |
| SELEC2024 | `summary/tag/name` | — |

In [2]:
def first_scalar(arr):
    """Return first element of array"""
    a = np.asarray(arr, dtype=float)
    return float(a.flat[0])


PLTH    = np.full(N, np.nan)          # loss power [W]
SPLASMA = np.full(N, np.nan)          # LCFS surface area [m^2]
BT      = np.full(N, np.nan)          # vacuum B_t at R0 [T]
NEL     = np.full(N, np.nan)          # line-averaged n_e [m^-3]
MEFF    = np.full(N, np.nan)          # effective hydrogenic mass [AMU]
IP      = np.full(N, np.nan)          # plasma current [A]
RGEO    = np.full(N, np.nan)          # geometric major radius [m]
TOK     = np.empty(N, dtype=object)   # tokamak name
DIVCON = np.empty(N, dtype=object)   # divertor config name
SEL     = np.zeros(N, dtype=bool)     # SELEC2024 high-density-branch flag

for i, pulse_dir in enumerate(pulse_dirs):
    uri = f"imas:hdf5?path={pulse_dir};pulse=0"
    if i % 20 == 0:
        print(f"Processing pulse {i}: {uri}")
    with imas.DBEntry(uri, "r") as entry:
        s = entry.get("summary")
        PLTH[i]    = first_scalar(s.global_quantities.power_loss.value)
        BT[i]      = np.abs(first_scalar(s.global_quantities.b0.value))
        NEL[i]     = first_scalar(s.line_average.n_e.value)
        MEFF[i]    = first_scalar(s.volume_average.meff_hydrogenic.value)
        IP[i]      = first_scalar(s.global_quantities.ip.value)
        RGEO[i]    = first_scalar(s.global_quantities.r0.value)
        TOK[i]     = str(s.machine).strip()
        SEL[i]     = str(s.tag.name).strip() == "SELEC2024=True" # Array of bools.

        eq = entry.get("equilibrium")
        SPLASMA[i] = first_scalar(eq.time_slice[0].global_quantities.surface)

        div = entry.get("divertors")
        DIVCON[i] = str(div.divertor[0].description).strip()
        
print("Done.")

print(f"  SELEC2024=True : {np.sum(SEL)}")

09:43:46 INFO     Parsing data dictionary version 4.1.1 @dd_zip.py:89


Processing pulse 0: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0000;pulse=0
Processing pulse 20: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0020;pulse=0
Processing pulse 40: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0040;pulse=0
Processing pulse 60: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0060;pulse=0
Processing pulse 80: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0080;pulse=0
Processing pulse 100: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0100;pulse=0
Processing pulse 120: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0120;pulse=0
Processing pulse 140: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0140;pulse=0
Processing pulse 160: imas:hdf5?path=c:\Users\curranf\IDStools\resources\results\tc26\pulse_0160;pulse=0
Processing pulse 180: imas:hdf5?path=c:\Users\curranf\IDStool

## Units, selection criteria.



In [3]:
Ploss_MW = PLTH / 1e6 # [MW]
ne_20    = NEL  / 1e20 # [10^20 m^-3]
ip_ma    = IP   / 1e6 # [MA]

VT_CONFIGS = {"VT", "CORNER"}#, "V5"} #,"V5E", "V5C", "V5L"}
D = np.where(np.isin(DIVCON, list(VT_CONFIGS)), 1.93, 1.0)

mask = SEL

print(f"Total pulses        : {N}")
print(f"SELEC2024=True      : {np.sum(SEL)}")
print(f"Complete + selected : {np.sum(mask)}")
print()
print("pre-selection:")
for tok in np.unique(TOK):
    print(f"  {tok}: {np.sum((TOK == tok))}")
print("SELEC2024=True")
for tok in np.unique(TOK[mask]):
    print(f"  {tok}: {np.sum((TOK == tok) & mask)}")

Total pulses        : 689
SELEC2024=True      : 482
Complete + selected : 482

pre-selection:
  AUG: 252
  CMOD: 77
  JET: 360
SELEC2024=True
  AUG: 162
  CMOD: 59
  JET: 261


## CMOD only table 4
CMOD has a discrepancy for the selection criteria. Paper states 58 pulses, dataset has 59 marked as SELEC2024=True. Expect small deviations, unless we can find the excluded datapoint by imposing equal fitted parameters?

Fitted coefficients differ, yet within error bars. 



In [4]:
from scipy.optimize import curve_fit

m = mask & (TOK == "CMOD") #& (ne_20 != 1.4)

P    = Ploss_MW[m]
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
Dfac = D[m]              # all 1.0 for C-Mod

def model(X, alpha, beta, eps):
    Bt, ne, Meff, S, Dfac = X
    return alpha * Bt**beta * ne**eps * (2.0/Meff) * Dfac * S

X = (Bt, ne, Meff, S, Dfac)

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.157 pm  0.034)")
print(f"beta  = {beta:+.3f} pm {perr[1]:.3f}   (paper -0.075 pm 0.112)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  1.10  pm 0.12)")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fitnorm: {rmse_fit:.3f} (paper  0.207)")




alpha = 0.1583 pm 0.0342   (paper  0.157 pm  0.034)
beta  = -0.080 pm 0.111   (paper -0.075 pm 0.112)
eps   = 1.101 pm 0.121   (paper  1.10  pm 0.12)
RMSE_fitnorm: 0.205 (paper  0.207)


## AUG only Table 4

Fitted parameters, and RMSE agree to the precision stated in the paper (three+ decimals), EXCEPT for the alpha (multiplicative) coefficient. Possibly, the dependent variable (i.e. P_LH/S) is scaled *uniformly* in the published dataset, compared to the one used in the paper? Too coincidental that all other parameters match.


In [5]:
m = mask & (TOK == "AUG")

P    = Ploss_MW[m] #* 0.925
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
Dfac = D[m]              # all 1.0 for AUG'
Dfac = np.ones_like(Bt) # force Dfac=1 for all pulses,

def model(X, alpha, beta, eps, zeta):
    Bt, ne, Meff, S, Dfac = X
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * Dfac * S

X = (Bt, ne, Meff, S, Dfac) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[-10, 0.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0314 pm  0.0042)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.689 pm 0.127)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  0.949  pm 0.0904)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  0.890  pm 0.064)")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.3f} (paper  0.142)")

alpha = 0.0339 pm 0.0046   (paper  0.0314 pm  0.0042)
beta  = 0.689 pm 0.127   (paper 0.689 pm 0.127)
eps   = 0.949 pm 0.090   (paper  0.949  pm 0.0904)
zeta  = 0.890 pm 0.064   (paper  0.890  pm 0.064)
RMSE_fit_norm: 0.142 (paper  0.142)


## JET ILW only TABLE 4

Similar to AUG. All parameters, as well as their uncertainties, and the RMSE match the results of the paper to stated precision, EXCEPT for the alpha coefficient.


In [6]:
m = mask & (TOK == "JET")

P    = Ploss_MW[m]# * 1.08135
Bt   = np.abs(BT[m]) #* 1.08135
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
is_VT = np.isin(DIVCON[m], list(VT_CONFIGS))   

#  Dfac = D[m]              D is a free parameter here

def model(X, alpha, beta, eps, zeta, Dfac):
    Bt, ne, Meff, S, is_VT = X
    D = np.where(is_VT, Dfac, 1.0)
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * D * S

X = (Bt, ne, Meff, S, is_VT) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.07, 0.4, 1.2, 1.1, 1.7],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta, Dfac = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0687 pm  0.0053)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.474 pm 0.051)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  1.22  pm 0.043)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  1.11  pm 0.04)")
print(f"Dfac  = {Dfac:.3f} pm {perr[4]:.3f}   (paper  1.73  pm 0.04)")
P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.3f} (paper  0.207)")

alpha = 0.0635 pm 0.0050   (paper  0.0687 pm  0.0053)
beta  = 0.474 pm 0.051   (paper 0.474 pm 0.051)
eps   = 1.221 pm 0.043   (paper  1.22  pm 0.043)
zeta  = 1.113 pm 0.038   (paper  1.11  pm 0.04)
Dfac  = 1.728 pm 0.036   (paper  1.73  pm 0.04)
RMSE_fit_norm: 0.207 (paper  0.207)


## Multi machine TC-26 (Bt) scaling

In [7]:
m = mask 

P    = Ploss_MW[m]# * 1.08135
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
is_VT = np.isin(DIVCON[m], list(VT_CONFIGS))   

def model(X, alpha, beta, eps, zeta, Dfac):
    Bt, ne, Meff, S, is_VT = X
    D = np.where(is_VT, Dfac, 1.0)
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * D * S

X = (Bt, ne, Meff, S, is_VT) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta, Dfac = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0441 pm  0.0025)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.580 pm 0.039)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  1.08  pm 0.03)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  0.975  pm 0.032)")
print(f"Dfac  = {Dfac:.3f} pm {perr[4]:.3f}   (paper  1.93  pm 0.04)")
P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.3f} (paper  0.238)")

alpha = 0.0451 pm 0.0026   (paper  0.0441 pm  0.0025)
beta  = 0.577 pm 0.039   (paper 0.580 pm 0.039)
eps   = 1.079 pm 0.026   (paper  1.08  pm 0.03)
zeta  = 0.974 pm 0.032   (paper  0.975  pm 0.032)
Dfac  = 1.931 pm 0.038   (paper  1.93  pm 0.04)
RMSE_fit_norm: 0.238 (paper  0.238)


## TC-26(Bt,S) scaling

In [8]:
def model(X, alpha, beta, eps, zeta, Dfac, eta):
    Bt, ne, Meff, S, is_VT = X
    D = np.where(is_VT, Dfac, 1.0)
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * D * S**eta

X = (Bt, ne, Meff, S, is_VT) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta, Dfac, eta = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0300 pm  0.0021)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.566 pm 0.039)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  1.27  pm 0.03)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  0.996  pm 0.031)")
print(f"Dfac  = {Dfac:.3f} pm {perr[4]:.3f}   (paper  1.81  pm 0.04)")
print(f"eta   = {eta:.3f} pm {perr[5]:.3f}   (paper  1.14  pm 0.02)")
P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.4f} (paper  0.220)")

alpha = 0.0303 pm 0.0022   (paper  0.0300 pm  0.0021)
beta  = 0.564 pm 0.039   (paper 0.566 pm 0.039)
eps   = 1.274 pm 0.034   (paper  1.27  pm 0.03)
zeta  = 0.995 pm 0.031   (paper  0.996  pm 0.031)
Dfac  = 1.806 pm 0.037   (paper  1.81  pm 0.04)
eta   = 1.141 pm 0.016   (paper  1.14  pm 0.02)
RMSE_fit_norm: 0.2193 (paper  0.220)


## TC-26(B_t, I_p) scaling

In [9]:
ip = np.abs(ip_ma[m])

def model(X, alpha, beta, eps, zeta, Dfac, gamma):
    Bt, ne, Meff, S, is_VT, ip = X
    D = np.where(is_VT, Dfac, 1.0)
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * D * S * ip**gamma

X = (Bt, ne, Meff, S, is_VT, ip) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta, Dfac, gamma = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0585 pm  0.0034)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.390 pm 0.042)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  1.22  pm 0.03)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  1.01  pm 0.03)")
print(f"Dfac  = {Dfac:.3f} pm {perr[4]:.3f}   (paper  1.73  pm 0.04)")
print(f"gamma = {gamma:.3f} pm {perr[5]:.3f}   (paper  0.233  pm 0.019)")
P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.4f} (paper  0.201)")

alpha = 0.0582 pm 0.0035   (paper  0.0585 pm  0.0034)
beta  = 0.389 pm 0.042   (paper 0.390 pm 0.042)
eps   = 1.219 pm 0.028   (paper  1.22  pm 0.03)
zeta  = 1.012 pm 0.031   (paper  1.01  pm 0.03)
Dfac  = 1.729 pm 0.036   (paper  1.73  pm 0.04)
gamma = 0.233 pm 0.019   (paper  0.233  pm 0.019)
RMSE_fit_norm: 0.2012 (paper  0.201)


## TC-26(B_p)

In [10]:
R = RGEO[m]
ip = np.abs(IP[m])
mu_0 = sp.constants.mu_0
Bpol = 2*np.pi * mu_0 * ip * R / S

def model(X, alpha, eps, zeta, Dfac, delta):
    Bt, ne, Meff, S, is_VT, Bpol = X
    D = np.where(is_VT, Dfac, 1.0)
    return alpha * ne**eps * (2.0/Meff)**zeta * D * S * Bpol**delta

X = (Bt, ne, Meff, S, is_VT, Bpol) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, eps, zeta, Dfac, delta = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.166 pm  0.006)")
print(f"eps   = {eps:.3f} pm {perr[1]:.3f}   (paper  1.07  pm 0.02)")
print(f"zeta  = {zeta:.3f} pm {perr[2]:.3f}   (paper  1.01  pm 0.03)")
print(f"Dfac  = {Dfac:.3f} pm {perr[3]:.3f}   (paper  1.67  pm 0.04)")
print(f"delta = {delta:.3f} pm {perr[4]:.3f}   (paper  0.631  pm 0.033)")
P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.4f} (paper  0.189)")

alpha = 0.1647 pm 0.0044   (paper  0.166 pm  0.006)
eps   = 1.072 pm 0.021   (paper  1.07  pm 0.02)
zeta  = 1.008 pm 0.030   (paper  1.01  pm 0.03)
Dfac  = 1.668 pm 0.035   (paper  1.67  pm 0.04)
delta = 0.631 pm 0.033   (paper  0.631  pm 0.033)
RMSE_fit_norm: 0.1887 (paper  0.189)


## Reproduce TABLE 2 from paper.
Matches Table 2 exactly, EXCEPT for the noted discrepancy (N=59, rather than 58 for AUG), *and* 

In [11]:
for tok in np.unique(TOK[mask]):
    m = mask & (TOK == tok)
    ne_19 = NEL[m] / 1e19
    Ip_MA = np.abs(IP[m]) / 1e6
    print(rf"{tok} (N={np.sum(m)}):")
    print(rf"  Bt  : {np.min(np.abs(BT[m])):.2f} - {np.max(np.abs(BT[m])):.2f} [T]")
    print(rf"  Ip  : {np.min(Ip_MA):.2f} - {np.max(Ip_MA):.2f} [MA]")
    print(rf"  ne  : {np.min(ne_19):.2f} - {np.max(ne_19):.2f} [10^19 m^-3]")
    print(rf"  S   : {np.mean(SPLASMA[m]):.2f} [m^2]")


AUG (N=162):
  Bt  : 1.69 - 2.98 [T]
  Ip  : 0.60 - 1.01 [MA]
  ne  : 2.58 - 6.79 [10^19 m^-3]
  S   : 43.80 [m^2]
CMOD (N=59):
  Bt  : 5.27 - 7.67 [T]
  Ip  : 0.63 - 1.78 [MA]
  ne  : 14.00 - 28.70 [10^19 m^-3]
  S   : 7.37 [m^2]
JET (N=261):
  Bt  : 1.29 - 3.68 [T]
  Ip  : 1.23 - 3.94 [MA]
  ne  : 1.63 - 6.42 [10^19 m^-3]
  S   : 138.95 [m^2]


## Correlation matrix

In [18]:
m = mask #& (ne_20 != 1.4)

S = SPLASMA[m]
Bt   = np.abs(BT[m])
ne   = ne_20[m]
R = RGEO[m]
ip = np.abs(IP[m])
mu_0 = sp.constants.mu_0
Bpol = 2*np.pi * mu_0 * ip * R / S

labels = ["ne", "Bt", "ip", "Bpol"]
data = np.vstack([ne, Bt, ip, Bpol])
corr = np.corrcoef(data)

print(corr)
print()
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        print(f"corr({labels[i]}, {labels[j]}) = {corr[i, j]:.4f}")

[[ 1.          0.90013536 -0.26276659  0.88760183]
 [ 0.90013536  1.         -0.12632845  0.89697358]
 [-0.26276659 -0.12632845  1.          0.13288615]
 [ 0.88760183  0.89697358  0.13288615  1.        ]]

corr(ne, Bt) = 0.9001
corr(ne, ip) = -0.2628
corr(ne, Bpol) = 0.8876
corr(Bt, ip) = -0.1263
corr(Bt, Bpol) = 0.8970
corr(ip, Bpol) = 0.1329
